# Notebook 0 — Visite guidée du banc d'essai

Ce notebook fait deux choses : présenter **l'architecture cible** du projet, module par
module, et montrer **le premier morceau de code réel** — l'interface de chargement des
jeux d'évaluation et le loader de référence (BEIR/SciFact) — en l'exécutant pour de vrai.

Règle de lecture valable pour tous les notebooks du projet : **rien n'est réimplémenté
ici**. Tout ce qui s'exécute est importé depuis `src/llm_testbench/`, affiché avec
`inspect.getsource()`, puis lancé. Ce que vous lisez est ce qui tourne.

Deux modes d'exécution :
- **complet** (défaut) : télécharge le vrai SciFact (~4,5 MB, une seule fois) ;
- **échantillon** (`LLM_TESTBENCH_SAMPLE=1`, ce que fait la CI) : fixtures locales
  minuscules, zéro réseau, aucun chiffre publiable.

## Le problème — mesurer un système LLM sans utilisateurs

La solution naïve pour évaluer un RAG : poser dix questions à la main, trouver les
réponses « plutôt bonnes », passer à la suite. Ça ne détecte ni les régressions, ni les
implémentations fausses, ni les gains imaginaires — et ça ne se compare à rien.

Ce projet prend le parti inverse : **chaque technique est mesurée sur des jeux publics à
vérité de référence annotée**, et comparée aux scores publiés. L'intérêt est pédagogique
avant tout : si notre BM25 sort très en dessous de la littérature sur SciFact, notre
implémentation est fausse **et on le sait le jour même**. Sur un corpus maison, aucun
repère — on peut optimiser un système bancal pendant trois semaines sans s'en apercevoir.

Deux limites assumées, écrites noir sur blanc dans le README :
1. **Pas d'utilisateurs ni de trafic réel** — tout ce qui suppose de la production sera
   simulé par un générateur de trafic synthétique (phase 5).
2. **Contamination** : ces jeux sont probablement dans les données d'entraînement des
   modèles évalués, donc les scores sont flattés. C'est un objet de mesure (phase 2),
   pas un détail qu'on ignore.

In [1]:
# --- Cellules de configuration : modifiez, puis « Run All » ---
import os

# Mode échantillon : fixtures locales, zéro réseau (c'est le mode CI).
SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"

# En mode échantillon : nombre maximal de requêtes conservées.
MAX_QUERIES = 5

# Split chargé plus bas : "test" ou "train".
SPLIT = "test"

# Index de la requête détaillée dans les sorties intermédiaires.
SHOW_QUERY_INDEX = 0

print(f"mode échantillon : {SAMPLE}")

mode échantillon : False


## L'architecture cible

Un seul système, qui grossit à chaque phase — jamais un second projet.

```
src/
├── app/          # API FastAPI : /chat avec citations et refus explicite   (phase 1)
├── llm/          # abstraction multi-provider, retries, cache, budgets     (phase 1)
├── sources/      # interface Source générique : docs, SQL, API, images     (phases 1, 4)
├── ingest/       # parsing, chunking, embeddings, indexation pgvector     (phases 1, 3)
├── retrieval/    # recherche dense, BM25, hybride, re-ranking             (phases 1, 3)
├── generation/   # assemblage de contexte, chaînes                        (phase 1)
├── agents/       # orchestration, outils, garde-fous, MCP                 (phase 4)
├── eval/         # types, loaders, harness, métriques, juges  ← ON EST ICI (phases 0, 2, 4)
└── obs/          # traces OTel/Langfuse, coût par requête                  (phases 1, 5)
```

La phase 0 ne livre que le coin en bas : `eval/types.py`, `eval/loaders/`, plus les
fondations invisibles ici — outillage, CI, cluster k3d avec Postgres/pgvector, bootstrap
Terraform (bucket d'état + alerte de budget, ADR 002). Pourquoi commencer par `eval/` ?
Parce que le brief fait de **la mesure** l'argument central du projet : tout ce que les
autres modules produiront sera jugé par ce module-là.

In [2]:
import inspect
import statistics

from llm_testbench import __version__
from llm_testbench.config import Settings, find_repo_root
from llm_testbench.eval import types as eval_types
from llm_testbench.eval.loaders import BeirLoader
from llm_testbench.eval.loaders import beir as beir_module

ROOT = find_repo_root()
settings = Settings(
    data_dir=ROOT / "data" / "cache",
    fixtures_dir=(ROOT / "tests" / "fixtures") if SAMPLE else None,
    sample=SAMPLE,
)
print(f"llm_testbench {__version__} — racine : {ROOT}")
print(f"source de données : {'fixtures locales' if SAMPLE else 'Hugging Face (cache data/cache/)'}")

llm_testbench 0.1.0 — racine : /Users/christophe/Sites/roadmap-ai-engineer
source de données : Hugging Face (cache data/cache/)


## Les options — pourquoi des loaders maison

Deux approches existaient (détail : ADR 004) :

1. **Réutiliser un cadre existant** — `ir_datasets` ou le paquet `beir` savent charger
   les jeux de retrieval. Mais ils ne couvrent que le retrieval (pas les familles QA,
   SQL, images qui arrivent), imposent leurs types, et masquent précisément la mécanique
   que ce projet veut exposer : d'où viennent les données, ce qu'on joint avec quoi,
   ce qui casse.
2. **Types maison validés + loaders par famille** — plus de code à nous, mais un contrat
   unique : provenance déclarée, intégrité vérifiée à la construction, mode échantillon
   explicite.

Arbitrage : option 2. Le harness de la phase 2 consommera ces types tels quels, et on
comparera alors notre harness à Ragas/promptfoo/DeepEval — après avoir construit le nôtre.

## Lecture guidée 1/3 — les types (`src/llm_testbench/eval/types.py`)

Le vocabulaire est celui de la littérature IR — *corpus*, *queries*, *qrels* — gardé tel
quel pour que nos scores restent comparables aux papiers. Trois briques d'abord :

In [3]:
for cls in (eval_types.Document, eval_types.Query, eval_types.DatasetInfo):
    print(inspect.getsource(cls))

class Document(BaseModel):
    model_config = ConfigDict(frozen=True)

    doc_id: str
    text: str
    title: str = ""
    metadata: dict[str, str] = Field(default_factory=dict)

class Query(BaseModel):
    model_config = ConfigDict(frozen=True)

    query_id: str
    text: str
    metadata: dict[str, str] = Field(default_factory=dict)

class DatasetInfo(BaseModel):
    """Carte d'identité d'un jeu : de quoi tracer la provenance de chaque chiffre."""

    model_config = ConfigDict(frozen=True)

    key: str
    """Identifiant court utilisé partout dans le banc (ex. ``beir/scifact``)."""

    name: str
    homepage: str
    license: str
    """Licence telle qu'affichée par la source de distribution — voir la vérification
    du 2026-08-26 dans docs/ : pour BEIR, c'est la revendication de l'empaquetage,
    pas une garantie amont."""

    source: str
    """D'où les données sont effectivement chargées (dépôts Hugging Face, URL...)."""



Trois choix discrets mais importants :

- **`frozen=True`** : un jeu d'évaluation est immuable. Aucun code de retrieval ne peut
  « corriger » un document en douce, et les objets sont hashables.
- **Les ids sont des `str` partout**, même quand la source les livre en `int`. Un `4983`
  entier côté qrels qui ne matche pas un `"4983"` chaîne côté corpus, c'est un recall
  divisé par deux sans aucune erreur visible.
- **`DatasetInfo` voyage avec les données** : source exacte, licence *telle qu'affichée*
  (pour BEIR c'est la revendication de l'empaquetage, pas une garantie amont — vérifié
  le 2026-08-26), homepage. Chaque chiffre du futur tableau de résultats doit pouvoir
  être tracé jusqu'à sa provenance.

Le cœur du contrat, maintenant — le conteneur validé :

In [4]:
print(inspect.getsource(eval_types.RetrievalDataset))

class RetrievalDataset(BaseModel):
    model_config = ConfigDict(frozen=True)

    info: DatasetInfo
    split: str
    documents: tuple[Document, ...]
    queries: tuple[Query, ...]
    qrels: Qrels

    sampled: bool = False
    """True si le jeu a été tronqué (mode échantillon). Un jeu échantillonné vérifie
    la mécanique ; il ne produit JAMAIS un chiffre publiable."""

    @model_validator(mode="after")
    def _check_referential_integrity(self) -> "RetrievalDataset":
        doc_ids = {d.doc_id for d in self.documents}
        query_ids = {q.query_id for q in self.queries}

        if len(doc_ids) != len(self.documents):
            raise ValueError("doc_id en double dans le corpus")
        if len(query_ids) != len(self.queries):
            raise ValueError("query_id en double dans les requêtes")

        orphan_queries = set(self.qrels) - query_ids
        if orphan_queries:
            raise ValueError(
                f"qrels référencent des requêtes absentes : {sorted(orph

Le validateur `_check_referential_integrity` est la vraie raison d'être de cette classe.
Il refuse, **à la construction** :

- les ids en double (corpus ou requêtes) ;
- des qrels qui référencent une requête ou un document absents — la signature classique
  d'une jointure ratée ou d'un problème de types d'ids ;
- **une requête sans aucun jugement**. Celui-là est subtil : si le harness calcule un
  recall@k sur une requête que personne n'a jamais jugée, le score de cette requête est
  0 par construction et la moyenne du jeu est silencieusement fausse. Le loader doit
  filtrer ces requêtes ; s'il ne le fait pas, l'erreur est bruyante, ici, pas trois
  phases plus tard dans un chiffre bizarre.

Et le flag `sampled` : un jeu tronqué **se déclare lui-même** impropre à produire un
chiffre publiable. Ce n'est pas une convention d'équipe, c'est porté par les données.

## Lecture guidée 2/3 — le loader BEIR (`src/llm_testbench/eval/loaders/beir.py`)

La distribution BEIR sur Hugging Face a trois particularités (vérifiées le 2026-08-26) :
chaque jeu vit dans **deux dépôts** (`BeIR/scifact` pour corpus + requêtes,
`BeIR/scifact-qrels` pour les jugements), les dépôts sont en Parquet natif (chargement
sans script, compatible `datasets` ≥ 3), et le fichier queries contient **plus de
requêtes que le split n'en juge** — SciFact : 1 109 requêtes, 300 jugées en test.

In [5]:
print(inspect.getsource(BeirLoader))

class BeirLoader(RetrievalLoader):
    def __init__(self, dataset: str = "scifact", settings: Settings | None = None) -> None:
        if dataset not in BEIR_DATASETS:
            raise ValueError(
                f"jeu BEIR non branché : {dataset!r} (connus : {sorted(BEIR_DATASETS)})"
            )
        self._dataset = dataset
        self._settings = settings if settings is not None else get_settings()

    @property
    def info(self) -> DatasetInfo:
        return BEIR_DATASETS[self._dataset]

    def load(self, split: str = "test", max_queries: int | None = None) -> RetrievalDataset:
        corpus_rows, query_rows, qrels_rows = self._load_rows(split)
        return build_retrieval_dataset(
            info=self.info,
            split=split,
            corpus_rows=corpus_rows,
            query_rows=query_rows,
            qrels_rows=qrels_rows,
            max_queries=max_queries,
        )

    # ------------------------------------------------------------------ sources

  

Le loader ne fait que **choisir la source des lignes** : Hugging Face (avec cache local
dans `data/cache/`) ou fixtures JSON si `fixtures_dir` est défini. Toute l'intelligence
est dans la jointure, une fonction **pure** — la même qui tourne sur les fixtures de
tests et sur les données réelles. On ne teste pas un mock de la logique ; on rejoue la
vraie logique sur des données minuscules.

In [6]:
print(inspect.getsource(beir_module.build_retrieval_dataset))

def build_retrieval_dataset(
    info: DatasetInfo,
    split: str,
    corpus_rows: list[Row],
    query_rows: list[Row],
    qrels_rows: list[Row],
    max_queries: int | None = None,
) -> RetrievalDataset:
    """Joint corpus + requêtes + qrels en un jeu validé.

    Pure et indépendante de la source des lignes : c'est elle que les tests
    hors ligne couvrent, la même qui tourne sur les données réelles.
    """
    qrels: Qrels = {}
    for row in qrels_rows:
        # Les ids qrels arrivent parfois typés int côté HF : tout passe en str.
        query_id, doc_id = str(row["query-id"]), str(row["corpus-id"])
        qrels.setdefault(query_id, {})[doc_id] = int(row["score"])

    judged_queries = [
        Query(query_id=str(row["_id"]), text=row["text"])
        for row in query_rows
        if str(row["_id"]) in qrels
    ]

    sampled = max_queries is not None and len(judged_queries) > max_queries
    if sampled:
        judged_queries = judged_queries[:max_queries]
        kept

Bloc par bloc :

1. **qrels d'abord** — dictionnaire `query_id -> {doc_id: score}`, avec cast `str()`
   systématique : côté HF, les ids qrels de SciFact arrivent typés `int`.
2. **Filtrage des requêtes** : seules celles présentes dans les qrels du split survivent
   (1 109 → 300 en test). C'est la garantie exigée par le validateur vu plus haut.
3. **Échantillonnage** : les `max_queries` *premières* requêtes jugées (ordre du fichier,
   donc déterministe — pas de `random` : deux exécutions CI donnent le même jeu), et
   uniquement les documents qu'elles jugent. Le corpus n'a alors **plus de
   distracteurs** : la mécanique est vérifiable, un score de retrieval n'y voudrait
   rien dire — d'où `sampled=True`.

## Exécution — avec les sorties intermédiaires visibles

In [7]:
loader = BeirLoader("scifact", settings=settings)
dataset = loader.load(SPLIT, max_queries=MAX_QUERIES if SAMPLE else None)
dataset.describe()

{'dataset': 'beir/scifact',
 'split': 'test',
 'documents': 5183,
 'queries': 300,
 'judgments': 339,
 'sampled': False}

In [8]:
# La provenance voyage avec les données : de quoi tracer chaque futur chiffre.
dataset.info.model_dump()

{'key': 'beir/scifact',
 'name': 'SciFact (BEIR)',
 'homepage': 'https://github.com/allenai/scifact',
 'license': "cc-by-sa-4.0 (revendication de l'empaquetage BEIR, cf. ADR 001)",
 'source': 'hf:BeIR/scifact + hf:BeIR/scifact-qrels'}

In [9]:
# Une requête réelle, ses jugements, et le(s) document(s) jugés pertinents.
query = dataset.queries[SHOW_QUERY_INDEX]
print(f"requête {query.query_id!r} : {query.text}\n")
print(f"qrels bruts : {dataset.qrels[query.query_id]}\n")

for doc_id in sorted(dataset.relevant_doc_ids(query.query_id)):
    doc = dataset.document_by_id(doc_id)
    excerpt = doc.text[:280] + ("…" if len(doc.text) > 280 else "")
    print(f"--- doc {doc.doc_id} — {doc.title}")
    print(f"    {excerpt}\n")

requête '1' : 0-dimensional biomaterials show inductive properties.

qrels bruts : {'31715818': 1}

--- doc 31715818 — New opportunities: the use of nanotechnologies to manipulate and track stem cells.
    Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the int…



In [10]:
# Physionomie du jeu : longueurs de documents et densité des jugements.
doc_words = [len(d.text.split()) for d in dataset.documents]
judgments_per_query = [len(j) for j in dataset.qrels.values()]

print(f"documents : {len(doc_words)}")
print(
    "mots/document : "
    f"min {min(doc_words)}, médiane {statistics.median(doc_words):.0f}, max {max(doc_words)}"
)
print(
    "jugements/requête : "
    f"moyenne {statistics.mean(judgments_per_query):.2f}, max {max(judgments_per_query)}"
)

documents : 5183
mots/document : min 26, médiane 192, max 1524
jugements/requête : moyenne 1.13, max 5


### Les chiffres — contrôle contre la littérature

Le réflexe fondateur du projet : chaque fois qu'un chiffre peut être confronté à une
source publiée, on le fait, et **un écart est un bug, pas un résultat**. Ici, premier
contrôle trivial : les volumes du jeu chargé doivent être ceux du papier BEIR
(SciFact : 5 183 documents, 300 requêtes test, ~1,1 jugement pertinent par requête).
La cellule suivante vérifie — et échoue bruyamment en cas d'écart, y compris sous nbmake
dans la CI… sauf en mode échantillon, où les chiffres sont par définition non publiables.

In [11]:
if dataset.sampled or SAMPLE:
    print("Mode échantillon : chiffres NON publiables, contrôle littérature sans objet.")
    print(dataset.describe())
else:
    expected = {"documents": 5183, "queries": 300}
    actual = dataset.describe()
    for key, value in expected.items():
        assert actual[key] == value, f"{key} : {actual[key]} != {value} (papier BEIR)"
    print("OK — volumes conformes au papier BEIR :", {k: actual[k] for k in expected})
    print("Repère pour les phases 2-3 : BM25 fait ~0,67 de nDCG@10 sur SciFact")
    print("(papier BEIR, table 2). C'est la baseline que notre implémentation devra")
    print("retrouver — très en dessous = notre code est faux, pas le jeu.")

OK — volumes conformes au papier BEIR : {'documents': 5183, 'queries': 300}
Repère pour les phases 2-3 : BM25 fait ~0,67 de nDCG@10 sur SciFact
(papier BEIR, table 2). C'est la baseline que notre implémentation devra
retrouver — très en dessous = notre code est faux, pas le jeu.


### Ce que SciFact mesure — et ce qu'il ne mesure pas

**Mesure** : la capacité d'un système de retrieval à retrouver, parmi 5 183 abstracts
scientifiques, celui ou ceux qui **soutiennent ou réfutent** une affirmation scientifique
(« les statines augmentent les myalgies »). Requêtes formulées comme des affirmations,
pas des questions — un bon test de robustesse pour les embeddings entraînés sur des
paires question/réponse.

**Ne mesure pas** : la génération (aucune réponse rédigée attendue), le multi-hop (un
seul document suffit presque toujours), le refus (toute requête a sa vérité), le
dialogue. Petit corpus : les effets d'échelle d'index n'apparaîtront pas. C'est un jeu
de **validation de mécanique de retrieval** — exactement le rôle qu'on lui donne.

## Le reste du banc — famille par famille

Chaque entrée a été vérifiée le 2026-08-26 (`docs/datasets-verification-2026-08-26.md`,
ADR 001). Colonne « piège » : ce qu'on a appris en vérifiant, pas ce que disent les
tutoriels.

### Retrieval et embeddings

| Jeu | Fournit | Mesurera | Piège vérifié |
|---|---|---|---|
| **SciFact** (BEIR) | 5,2k docs, 300 req., qrels ~binaires | recall@k, MRR, nDCG — validation d'implémentation | qrels dans un dépôt séparé ; requêtes non jugées à filtrer |
| **NFCorpus** (BEIR) | 3,6k docs, 323 req., **qrels gradués 0-2** | nDCG avec pertinence graduée | ~134k jugements : la moyenne des « pertinents » n'a pas le même sens qu'en binaire |
| **FiQA** (BEIR) | 57,6k docs finance | passage à l'échelle ×10, jargon métier | 6× plus gros — itérations plus lentes |
| **MTEB** `MTEB(fra, v1)` | 22 tâches françaises standardisées | comparaison de modèles d'embeddings (phase 1) | API v2 cassante : tout tutoriel pré-2025 est faux |

### QA : multi-hop, sans réponse, documents bruts

| Jeu | Fournit | Mesurera | Piège vérifié |
|---|---|---|---|
| **HotpotQA** | 90k questions 2 sauts + passages de preuve | décomposition multi-hop (phase 3) | réponses du test cachées → évaluer sur validation |
| **SQuAD 2.0** | 142k questions dont ~50k **sans réponse** | taux de refus correct vs refus abusif | pas de flag explicite côté HF : non-répondable = liste de réponses vide |
| **QASPER** | 1 585 articles **entiers** + spans de preuve | stratégies de chunking sur documents bruts (phase 3) | ne se charge que via la branche `refs/convert/parquet` |
| MuSiQue, 2WikiMultihopQA | multi-hop plus dur / plus propre | en réserve si HotpotQA sature | distribution Google Drive/Dropbox + miroirs non officiels : provenance à épingler |

(**Natural Questions retiré** : ~145 GB pour un signal « sans réponse » indirect —
SQuAD 2.0 couvre le besoin. ADR 001.)

### Text-to-SQL et agents

| Jeu | Fournit | Mesurera | Piège vérifié |
|---|---|---|---|
| **Spider 1.0** | 10k questions, 200 bases **SQLite incluses** | exactitude d'exécution SQL (phase 4) | bases uniquement sur Google Drive ; contamination massive documentée ; leaderboard fermé |
| **BIRD Mini-Dev** | 500 instances SELECT, 11 bases, **PostgreSQL fourni** | text-to-SQL sur bases sales | le jeu complet fait 33 GB — exclu ; le Mini-Dev réduit notre travail de conversion |
| **tau2-bench (τ³)** | environnements complets : outils + règles + utilisateur simulé | agents multi-tours (phase 4) | **remplace τ-bench, officiellement déprécié** ; coût double (LLM agent + LLM utilisateur) ; chiffres du papier 2024 non comparables |
| **BFCL v4** | appels d'outils à granularité fine | exactitude des arguments, appels parallèles, « ne rien appeler » | harness pinné **Python 3.10** → environnement séparé ; scores comparables uniquement à version BFCL égale |
| **AgentDojo** | 629 cas d'injection en contexte agentique | sécurité des outils (phase 4.7) | API pré-1.0, instable par design |

### Tableaux, images, séries temporelles

| Jeu | Fournit | Mesurera | Piège vérifié |
|---|---|---|---|
| **TAT-QA** | 16,5k questions texte+tableau financiers, 17 MB | raisonnement numérique hybride (phase 4) | le plus propre de sa famille ; vérité du test publiée depuis 2024 |
| WikiTableQuestions | 22k questions sur tables | agrégations déterministes | dépôt HF à script mort → passer par GitHub ; licence CC BY-**SA** (la carte HF se trompe) |
| FinQA | ~8,3k questions calculs financiers | l'arithmétique est faite par du code, jamais par le modèle | bug pré-2022 dans le code officiel qui gonflait des scores publiés |
| **DocVQA** | 50k questions sur documents scannés | QA multimodale (phase 4.5) | site officiel : inscription + TLS cassé → miroirs HF ; licence d'origine floue ; évaluer sur validation |
| **ChartQA** | 32,7k questions sur graphiques | lecture de graphiques | licence **GPL-3.0** ; séparer splits humain/machine comme la littérature |
| **CORD v2** | 1 000 reçus, champs étiquetés | extraction structurée champ par champ | 1k publiés sur les 11k du papier → comparabilité partielle |
| FUNSD | 199 formulaires annotés | extraction sur formulaires | licence **non commerciale**, images jamais redistribuées ; annotations bruitées connues |
| Séries temporelles | **générées par nous**, vérité pandas | agrégations, fuseaux, valeurs manquantes | le générateur est un livrable versionné (phase 4) — zéro contamination possible, par construction |

## Expérimentations guidées

Modifiez les cellules de configuration en haut, puis « Run All » :

1. **`SPLIT = "train"`** — le split d'entraînement de SciFact : d'autres requêtes,
   d'autres qrels, aucun recouvrement avec le test (la cellule suivante le montre).
2. **`MAX_QUERIES = 2`** (en mode échantillon) — observez `sampled: True` et le corpus
   qui se réduit aux seuls documents jugés.
3. **`SHOW_QUERY_INDEX = 1` ou `2`** — d'autres requêtes dans les sorties intermédiaires.

In [12]:
# Test vs train : deux ensembles de requêtes disjoints.
other_split = "train" if SPLIT == "test" else "test"
other = loader.load(other_split, max_queries=MAX_QUERIES if SAMPLE else None)

current_ids = {q.query_id for q in dataset.queries}
other_ids = {q.query_id for q in other.queries}
print(f"{SPLIT} : {dataset.describe()}")
print(f"{other_split} : {other.describe()}")
print(f"requêtes communes aux deux splits : {len(current_ids & other_ids)}")

test : {'dataset': 'beir/scifact', 'split': 'test', 'documents': 5183, 'queries': 300, 'judgments': 339, 'sampled': False}
train : {'dataset': 'beir/scifact', 'split': 'train', 'documents': 5183, 'queries': 809, 'judgments': 919, 'sampled': False}
requêtes communes aux deux splits : 0


In [13]:
# Un loader refuse ce qu'il ne connaît pas — bruyamment, avec la liste de ce qu'il connaît.
try:
    BeirLoader("nfcorpus")
except ValueError as error:
    print(f"ValueError : {error}")

ValueError : jeu BEIR non branché : 'nfcorpus' (connus : ['scifact'])


## Questions d'entretien

<details><summary><b>1. Pourquoi évaluer sur des jeux publics plutôt que sur son propre corpus ?</b></summary>

Les scores publiés fournissent un repère objectif : une implémentation très en dessous
de la littérature est fausse, et on le sait immédiatement. Sur un corpus maison, aucun
repère — et le coût d'annotation est énorme. Contrepartie à toujours mentionner : la
contamination (ces jeux sont dans les données d'entraînement des modèles), donc scores
flattés ; se mesure avec un jeu « frais » de comparaison. Un corpus maison reste utile
pour démontrer la *méthode* de construction d'une évaluation (notre golden set, phase 2).
</details>

<details><summary><b>2. Qu'est-ce qu'un qrel ? Binaire vs gradué — qu'est-ce que ça change ?</b></summary>

Un jugement de pertinence : pour une requête, quels documents sont pertinents, à quel
degré. Binaire (SciFact) : recall@k et MRR suffisent. Gradué (NFCorpus, 0-2) : il faut
une métrique sensible au degré, nDCG typiquement ; un recall binaire y jetterait de
l'information et le seuil « pertinent si ≥ 1 » devient un choix à documenter.
</details>

<details><summary><b>3. Le fichier queries contient des requêtes jamais jugées. Que se passe-t-il si on ne les filtre pas ?</b></summary>

Toute métrique moyennée sur les requêtes est diluée : une requête sans qrels score 0 par
construction (rien à retrouver), la moyenne baisse silencieusement, sans aucune erreur.
SciFact : 1 109 requêtes dans le fichier, 300 jugées en test — sans filtrage, tout score
serait divisé par ~3,7. D'où la validation à la construction : requête sans jugement =
exception immédiate.
</details>

<details><summary><b>4. Comment garantir qu'un chiffre de CI (échantillon) ne sera jamais présenté comme un résultat ?</b></summary>

Le flag est porté par les données elles-mêmes (`sampled=True`), posé par le loader au
moment de la troncature — pas par une convention d'équipe ni une variable d'ambiance. Le
harness (phase 2) refusera d'archiver un run sur jeu échantillonné. Par ailleurs le
corpus échantillonné n'a plus de distracteurs : tout score y serait absurde, autant le
rendre impossible à publier.
</details>

<details><summary><b>5. Pourquoi caster tous les ids en chaînes ?</b></summary>

Les sources sont incohérentes : SciFact livre les ids du corpus en `str` et ceux des
qrels en `int`. Une jointure `4983 == "4983"` échoue en silence en Python : pas
d'erreur, juste des qrels orphelins et un recall faux. On normalise à la frontière
(dans la jointure) et la validation d'intégrité attrape toute fuite résiduelle.
</details>

<details><summary><b>6. Vos scores sur SciFact battent le papier BEIR. Réaction ?</b></summary>

Suspicion d'abord, célébration jamais : un écart important — dans les deux sens — est un
bug jusqu'à preuve du contraire. Pistes : fuite du split (évaluation sur des requêtes de
train), métrique mal implémentée (nDCG sans normalisation idéale), contamination du
modèle d'embeddings, corpus tronqué. La phase 2 ajoute les intervalles de confiance par
bootstrap : sans eux, un « +2 % » a de bonnes chances de ne pas exister.
</details>

## Ce qu'on n'a pas fait, et pourquoi

- **Les autres loaders** (NFCorpus, HotpotQA, QASPER, Spider…) : chaque phase branche
  ceux dont elle a besoin. La semaine 1 contient déjà toute l'infra ; le contrat
  (types + validation + fixtures + tests) est posé, les ajouts sont mécaniques.
- **Aucune métrique de retrieval** (recall@k, nDCG…) : c'est le cœur de la phase 2,
  la phase centrale du projet. Ici on ne livre que des données propres à mesurer.
- **Pas d'épinglage de révision HF** (`revision=...`) dans le loader : les dépôts BeIR
  sont stables depuis des années. Si un futur `describe()` dévie des volumes du papier,
  le contrôle littérature ci-dessus le criera — on épinglera à ce moment-là.
- **Langfuse et le cloud AWS** : reportés en phase 1 (ADR 002) — rien à tracer et rien
  à déployer tant qu'il n'y a ni appel LLM ni API. Seuls le bucket d'état Terraform et
  l'alerte de budget existent déjà.
- **Le générateur de séries temporelles** : phase 4, avec les agents qui le
  consommeront.